In [1]:
from PIL import Image
import torch
from torchvision import transforms, datasets
import math

In [5]:
torch.manual_seed(42)

## ゼロパディング

In [ ]:
def zeroPadding(image, padding=0):
    """
    Args:
        kernel_shape : (N, C, H, W)
    """
    N, C, H, W = image.shape
    
    # paddingが0以下なら処理を終了する
    if padding <= 0 :
        return image
    
    padding_image = torch.zeros(N, C, H + padding * 2, W + padding * 2, dtype=torch.float)
    
    padding_image[:, :, padding : -padding , padding : -padding] = image
    
    return padding_image

## 出力画像サイズの計算処理

In [6]:
def convN(x, kernel, padding, stride):
    """カーネルをずらし、加重和を計算する回数を計算(カーネルで覆えない部分は除外)
    出力画像サイズを返すことと同義
    """
    cn = (((x + padding * 2) - kernel) / stride) + 1
    return math.floor(cn) # カーネル外を除外

## 畳み込みの順伝搬

In [3]:
### 2次元のフィルターを実装する(N=1, C=1, padding=可変, stride=可変)
def conv2d(kernel_shape : tuple, image : torch.tensor, padding:int=1, stride:int=1, bias=False):
    """
    Args:
        kernel_shape : (out_channels, input_channels, H, W)
        image.shape : (N, input_channels, H, W)
        padding :
        stride : 
    """
    
    N, C, H, W = image.shape
    
    out_channels, input_channels, fH, fW = kernel_shape
    
    # 入力画像とカーネルのinput_channel一致チェック
    assert input_channels == C
    
    # カーネルとバイアスの定義
    kernel = torch.randint(0, 10, (kernel_shape), dtype=torch.float)
    b = torch.zeros((out_channels, 1, 1))
    
    # フィルターが縦に動く回数
    row_n = convN(H, fH, padding, stride)
    # フィルターが横に動く回数
    col_n = convN(W, fW, padding, stride)
    
    # 畳み込み結果行列
    conv_image = torch.zeros((N, out_channels, row_n, col_n))
    
    # パディング
    image = zeroPadding(image, padding)
    print(f"padding image \n {image}")
    print(f"kernel : \n {kernel}")
    
    # outputチャネル数
    for o in range(out_channels):
        # 1時畳み込み行列(input_kernel分の行列を保持)
        tmp_image = torch.zeros((N, C, row_n, col_n))
        # バッチサイズ数
        for n in range(N):
            # インプットチャネル数
            for c in range(C):
                # 画像の高さ
                for r in range(row_n):
                    # 画像の幅
                    for w in range(col_n):
                        clip_image = image[n, c, r * stride : r * stride +  fH, w * stride : w * stride + fW]
                        # ここでは、数学的な畳み込みではなく相互相関を計算している。本当の畳み込みはカーネルを反転する
                        # CNNでは、カーネルの値が学習によって求められるため、反転する計算コストを削減している
                        ckernel = kernel[o, c]
                        scalar = clip_image.flatten() @ ckernel.flatten()
                        print(f"========channel: {c}, row : {r}, width : {w}===========")
                        print(f"切り取り画像 \n {clip_image}")
                        print(f"計算結果 \n {scalar}")
                        tmp_image[n, c, r, w] = scalar

        outc = tmp_image.sum(dim=1) # チャンネルごとに畳み込み結果を加算
        conv_image[:, o] = outc # 出力チャネル1個分の畳み込み結果を格納
        print(f"入力チャネル数ごとの畳み込み : {outc}")
        
    # バイアスを加算
    if bias:
        conv_image = conv_image + b
        
    return conv_image

In [ ]:
n = 5
c = 3
s = 4
image = torch.arange(n*c*s*s, dtype=torch.float).reshape(n, c, s, s)
padding = 1
stride = 2
out_channels = 2

fH = 3 
fW = 3

kernel_shape = (out_channels, c, fH, fW)
print(f"画像表示")
print(image)
print(image.shape)
print(kernel_shape)

conv_image = conv2d(kernel_shape, image, padding, stride)
print(f"畳み込み結果 : \n {conv_image}")

## 畳み込みの逆伝搬時に求める勾配の種類

- カーネルの勾配
- 入力画像の勾配
- バイアスの勾配